In [1]:
import cv2
import numpy as np
import tensorflow as tf

In [2]:
model_path = "face_mask.h5"

model = tf.keras.models.load_model(
    model_path,
    compile=False
)

print("Model loaded successfully!")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)

Model loaded successfully!
Input shape: (None, 224, 224, 3)
Output shape: (None, 1)


In [3]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

print("Face detector loaded!")

Face detector loaded!


In [4]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Could not open webcam")
else:
    print("Webcam opened successfully!")

Webcam opened successfully!


In [ ]:
while True:
    
    ret, frame = cap.read()
    
    if not ret:
        print("Could not read frame")
        break

    # تبدیل تصویر به grayscale برای پیدا کردن صورت
    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    # پیدا کردن صورت‌ها
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(80, 80)
    )

    for (x, y, w, h) in faces:
        
        # بریدن قسمت صورت
        face = frame[y:y+h, x:x+w]

        # تبدیل اندازه به 224x224
        face = cv2.resize(
            face,
            (224, 224)
        )

        # تبدیل BGR به RGB
        face = cv2.cvtColor(
            face,
            cv2.COLOR_BGR2RGB
        )

        # تبدیل به float32
        face = face.astype("float32")

        # preprocessing مخصوص MobileNetV2
        face = tf.keras.applications.mobilenet_v2.preprocess_input(
            face
        )

        # اضافه کردن batch dimension
        face = np.expand_dims(
            face,
            axis=0
        )

        # پیش‌بینی
        prediction = model.predict(
            face,
            verbose=0
        )[0][0]

        # تعیین کلاس
        if prediction < 0.5:
            label = "With Mask"
            confidence = (1 - prediction) * 100
        else:
            label = "Without Mask"
            confidence = prediction * 100

        # رسم مستطیل دور صورت
        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            (0, 255, 0),
            2
        )

        # متن
        text = f"{label} {confidence:.1f}%"

        cv2.putText(
            frame,
            text,
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

    # نمایش تصویر
    cv2.imshow(
        "Face Mask Detection",
        frame
    )

    # با q خارج شو
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


cap.release()
cv2.destroyAllWindows()